# 🌟 2D → 3D AI Reconstruction Studio — Dual-AI Engine (Google Colab)

Hệ thống tái tạo mô hình 3D từ ảnh ứng dụng **2 Mô hình Trí Tuệ Nhân Tạo SOTA hàng đầu thế giới**:
- ⚡ **Chế độ 1 Ảnh (Single-View)**: Sử dụng **TripoSR** (Stability AI / VAST-AI) — ViT + Triplane NeRF + Marching Cubes, tạo khối 3D hoàn chỉnh chỉ trong **~1.5 giây**.
- 🌐 **Chế độ Đa Ảnh (Multi-View N ảnh)**: Sử dụng **DUSt3R** (NAVER LABS, CVPR 2024 Highlight) — CroCo Cross-Attention Transformer tự động định vị góc chụp camera và tối ưu đám mây điểm toàn cục không gian thực, xuất file `.glb` màu sắc sống động mà **không bị rỗng/khoét lẹm**.

---

In [ ]:
# ============================================================================
# CELL 1: Cài đặt Môi trường Dual-AI (Model 1: TripoSR | Model 2: DUSt3R)
# ============================================================================
import os, sys, glob, shutil

os.chdir('/content')
if not os.path.exists('/content/TripoSR'):
    print('📥 Đang clone TripoSR từ VAST-AI-Research (Model 1: Single-View AI)...')
    !git clone -q https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR

if not os.path.exists('/content/dust3r'):
    print('📥 Đang clone DUSt3R từ NAVER LABS (Model 2: Multi-View AI)...')
    !git clone -q --recursive https://github.com/naver/dust3r.git /content/dust3r

if not os.path.exists('/content/ImgToModel'):
    print('📥 Đang clone ImgToModel (Backend Dual-Engine Router)...')
    !git clone -q -b P6-FullStack-Cloud https://github.com/dduy26/Img2d-to-3d.git /content/ImgToModel

for p in ['/content/TripoSR', '/content/dust3r', '/content/dust3r/croco', '/content/ImgToModel']:
    if p not in sys.path:
        sys.path.insert(0, p)

print('📦 Đang cài đặt thư viện cần thiết cho cả 2 Model AI...')
!pip install -q --no-cache-dir --upgrade numpy scipy pillow einops omegaconf rembg trimesh transformers huggingface_hub scikit-image onnxruntime roma opencv-python
!pip install -q fastapi uvicorn python-multipart requests

# 1. Vá lỗi ufunc.__module__ trên NumPy Python 3.13 trực tiếp trên đĩa
for p in glob.glob('/usr/local/lib/python3*/dist-packages/numpy/_core/strings.py') + glob.glob('/usr/lib/python3*/dist-packages/numpy/_core/strings.py'):
    try:
        with open(p, 'r', encoding='utf-8') as f:
            _txt = f.read()
        if 'ufunc.__module__ = "numpy.strings"' in _txt:
            _txt = _txt.replace('ufunc.__module__ = "numpy.strings"', 'pass')
            _txt = _txt.replace('ufunc.__qualname__ = ufunc.__name__', 'pass')
            with open(p, 'w', encoding='utf-8') as f:
                f.write(_txt)
            print('🔧 Đã vá lỗi ufunc.__module__ trên NumPy Python 3.13 thành công!')
    except Exception:
        pass

# 2. Vá lỗi type hint _Ink trong Pillow (PIL._typing)
try:
    import typing, PIL._typing
    for _sym in ['_Ink', '_Coords', '_Point', '_Size']:
        if not hasattr(PIL._typing, _sym):
            setattr(PIL._typing, _sym, typing.Any)
except Exception:
    pass

import torch
print('=' * 65)
print(f'✅ PyTorch: {torch.__version__}')
print(f'✅ GPU Sẵn Sàng: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CẢNH BÁO: Đang chạy CPU, hãy đổi sang T4 GPU!"}')
print('=' * 65)


In [ ]:
# ============================================================================
# CELL 2: Lựa chọn Ảnh Đầu Vào (Đơn Ảnh 1-View Hoặc Đa Ảnh Multi-View)
# ============================================================================
import os, glob, shutil
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

INPUT_DIR = '/content/input'
OUTPUT_DIR = '/content/output'
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Xóa ảnh cũ trong input trước khi nạp mới
for f in glob.glob(f'{INPUT_DIR}/*'):
    if os.path.isfile(f):
        os.remove(f)

# Lựa chọn chế độ:
# - 'upload': Tự chọn ảnh từ máy tính (chọn 1 ảnh HOẶC chọn nhiều ảnh cùng lúc!)
# - 'sample_multiview_apple': Bộ 4 góc chụp chuẩn (Front, Right, Back, Left) từ Objaverse
# - 'sample_single_chair': Mẫu 1 ảnh đơn chuẩn TripoSR
INPUT_SOURCE = 'upload'  # @param ['upload', 'sample_multiview_apple', 'sample_single_chair']

if INPUT_SOURCE == 'upload':
    from google.colab import files
    print('📤 Hãy bấm nút "Choose Files" để tải ảnh từ máy tính (Chọn 1 ảnh hoặc nhiều ảnh):')
    uploaded = files.upload()
    for fname in uploaded.keys():
        shutil.move(fname, f'{INPUT_DIR}/{fname}')
elif INPUT_SOURCE == 'sample_multiview_apple':
    sample_dir = '/content/ImgToModel/tests/data/objaverse_train'
    for f in sorted(glob.glob(f'{sample_dir}/*.png')):
        shutil.copy2(f, f'{INPUT_DIR}/{os.path.basename(f)}')
    print('📸 Đã nạp bộ dữ liệu mẫu Đa Góc Nhìn (Multi-View 4 góc: Front, Right, Back, Left)!')
else:
    shutil.copy2('/content/TripoSR/examples/chair.png', f'{INPUT_DIR}/chair.png')
    print('📸 Đã nạp ảnh mẫu Đơn Ảnh (Single-View): chair.png')

input_paths = sorted(
    glob.glob(f'{INPUT_DIR}/*.png') + glob.glob(f'{INPUT_DIR}/*.jpg') + glob.glob(f'{INPUT_DIR}/*.jpeg') + glob.glob(f'{INPUT_DIR}/*.webp')
)
assert len(input_paths) > 0, '⚠️ Không tìm thấy ảnh nào trong /content/input/!'

print(f'✅ Tổng số ảnh đã sẵn sàng: {len(input_paths)} ảnh')
n_show = min(len(input_paths), 5)
fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 3.5))
if n_show == 1:
    axes = [axes]
for ax, p in zip(axes, input_paths[:n_show]):
    img = Image.open(p)
    ax.imshow(img)
    ax.set_title(f'{os.path.basename(p)}\n({img.size[0]}x{img.size[1]})', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
# CELL 3: Khởi Chạy Workflow Dual-AI Model (TripoSR cho 1 ảnh | DUSt3R cho N ảnh)
# ============================================================================
import sys, types, subprocess, os, glob, inspect, typing

# ----------------------------------------------------------------------------
# 0. SHIELD: Tương thích hoàn hảo NumPy 2.x & SciPy & Pillow trên Python 3.13
# ----------------------------------------------------------------------------
# A. Vá lỗi ufunc.__module__ trong strings.py trực tiếp trên đĩa
for p in glob.glob('/usr/local/lib/python3*/dist-packages/numpy/_core/strings.py') + glob.glob('/usr/lib/python3*/dist-packages/numpy/_core/strings.py'):
    try:
        with open(p, 'r', encoding='utf-8') as f:
            _txt = f.read()
        if 'ufunc.__module__ = "numpy.strings"' in _txt:
            _txt = _txt.replace('ufunc.__module__ = "numpy.strings"', 'pass')
            _txt = _txt.replace('ufunc.__qualname__ = ufunc.__name__', 'pass')
            with open(p, 'w', encoding='utf-8') as f:
                f.write(_txt)
            sys.modules.pop('numpy._core.strings', None)
    except Exception:
        pass

# B. Vá các hàm umath & _blas_supports_fpe bị thiếu trong NumPy 2.x
for _mod in ['numpy._core._multiarray_umath', 'numpy._core.umath']:
    try:
        _m = sys.modules.get(_mod) or __import__(_mod, fromlist=['*'])
        if not hasattr(_m, '_blas_supports_fpe'):
            setattr(_m, '_blas_supports_fpe', lambda *a, **kw: False)
        for _sym in ['_slice', '_center', '_expandtabs', '_expandtabs_length', '_ljust', '_rjust', '_zfill']:
            if not hasattr(_m, _sym):
                setattr(_m, _sym, lambda *a, **kw: None)
    except Exception:
        pass

# C. Vá lỗi thiếu type hint _Ink trong Pillow (PIL._typing)
try:
    import PIL._typing
    for _sym in ['_Ink', '_Coords', '_Point', '_Size']:
        if not hasattr(PIL._typing, _sym):
            setattr(PIL._typing, _sym, typing.Any)
except Exception:
    pass

# ----------------------------------------------------------------------------
# 1. Tự động kiểm tra và cài đặt bổ sung nếu môi trường thiếu thư viện cốt lõi
# ----------------------------------------------------------------------------
for _pkg in ['trimesh', 'scikit-image', 'einops', 'omegaconf', 'rembg', 'roma']:
    try:
        __import__(_pkg.replace('-', '_'))
    except Exception:
        print(f'📦 Đang tự động bổ sung thư viện thiếu: {_pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _pkg])

import time, torch, shutil, numpy as np
from PIL import Image
import trimesh

# 2. Marching Cubes thuần túy (Scikit-image cho TripoSR - tương thích mọi Python)
from skimage.measure import marching_cubes as _sk_mc
mod = types.ModuleType('torchmcubes')
def _mc_skimage(density, threshold):
    dev = density.device if hasattr(density, 'device') else 'cpu'
    d_np = density.detach().cpu().numpy() if hasattr(density, 'detach') else np.asarray(density)
    verts, faces, _, _ = _sk_mc(d_np, level=float(threshold))
    return torch.from_numpy(verts.astype(np.float32)).to(dev), torch.from_numpy(faces.astype(np.int64)).to(dev)
mod.marching_cubes = _mc_skimage
sys.modules['torchmcubes'] = mod

# 3. Fallback rembg / onnxruntime nếu thiếu model phụ trợ
try:
    import onnxruntime, rembg
except Exception:
    rembg_dummy = types.ModuleType('rembg')
    rembg_dummy.new_session = lambda *a, **kw: None
    rembg_dummy.remove = lambda img, *a, **kw: img.convert('RGBA')
    sys.modules['rembg'] = rembg_dummy

# 4. Key Remapper tự động cho ViT Transformer (TripoSR)
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground
_orig_load_state_dict = TSR.load_state_dict
def _patched_load_state_dict(self, state_dict, strict=True, assign=False):
    model_keys = self.state_dict().keys()
    needs_remap = any(k.startswith('image_tokenizer.model.layers.') for k in model_keys)
    has_old = any(k.startswith('image_tokenizer.model.encoder.layer.') for k in state_dict.keys())
    if needs_remap and has_old:
        new_dict = {}
        for k, v in state_dict.items():
            if k.startswith('image_tokenizer.model.encoder.layer.'):
                nk = k.replace('image_tokenizer.model.encoder.layer.', 'image_tokenizer.model.layers.')
                nk = nk.replace('.attention.attention.query.', '.attention.q_proj.')
                nk = nk.replace('.attention.attention.key.', '.attention.k_proj.')
                nk = nk.replace('.attention.attention.value.', '.attention.v_proj.')
                nk = nk.replace('.attention.output.dense.', '.attention.o_proj.')
                nk = nk.replace('.intermediate.dense.', '.mlp.fc1.')
                nk = nk.replace('.output.dense.', '.mlp.fc2.')
                new_dict[nk] = v
            else:
                new_dict[k] = v
        state_dict = new_dict
    return _orig_load_state_dict(self, state_dict, strict=strict, assign=assign)
TSR.load_state_dict = _patched_load_state_dict

out_glb = f'{OUTPUT_DIR}/reconstructed_model.glb'
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
t0 = time.time()

print('=' * 75)
if len(input_paths) == 1:
    # ------------------------------------------------------------------------
    # CASE 1: SINGLE-VIEW AI MODEL (TripoSR - Stability AI)
    # ------------------------------------------------------------------------
    print('🔀 [ROUTER]: Phát hiện 1 ảnh ➔ Kích hoạt MODEL 1: TripoSR SOTA AI...')
    print('=' * 75)
    model_tripo = TSR.from_pretrained('stabilityai/TripoSR', config_name='config.yaml', weight_name='model.ckpt')
    model_tripo.renderer.set_chunk_size(8192)
    model_tripo.to(device)

    print('🔍 Đang tách nền tự động...')
    raw_img = Image.open(input_paths[0]).convert('RGB')
    try:
        clean_img = remove_background(raw_img, rembg.new_session())
    except Exception:
        clean_img = raw_img.convert('RGBA')
    clean_img = resize_foreground(clean_img, ratio=0.85)
    clean_arr = np.array(clean_img).astype(np.float32) / 255.0
    clean_arr = clean_arr[:, :, :3] * clean_arr[:, :, 3:4] + (1.0 - clean_arr[:, :, 3:4]) * 0.5
    clean_rgb = Image.fromarray((clean_arr * 255.0).astype(np.uint8))

    print('⚡ Đang suy luận mô hình 3D (ViT + Triplane NeRF + Marching Cubes)...')
    with torch.no_grad():
        scene_codes = model_tripo([clean_rgb], device=device)
        meshes = model_tripo.extract_mesh(scene_codes, True, resolution=256, threshold=25.0)
    mesh = meshes[0]
    mesh.export(out_glb)
    mode_name = 'Single-View AI (TripoSR)'
else:
    # ------------------------------------------------------------------------
    # CASE 2: MULTI-VIEW AI MODEL (DUSt3R - NAVER LABS CVPR 2024)
    # ------------------------------------------------------------------------
    print(f'🔀 [ROUTER]: Phát hiện {len(input_paths)} ảnh ➔ Kích hoạt MODEL 2: DUSt3R Multi-View AI!')
    print('=' * 75)
    from dust3r.model import AsymmetricCroCo3DStereo
    from dust3r.inference import inference
    from dust3r.image_pairs import make_pairs
    from dust3r.cloud_opt import global_aligner, GlobalAlignerMode
    from dust3r.utils.image import load_images
    from dust3r.demo import get_3D_model_from_scene
    import dust3r.viz

    # Vô hiệu hóa việc vẽ hình nón camera ảo (tránh lỗi IndexError và làm sạch mô hình)
    dust3r.viz.add_scene_cam = lambda *args, **kwargs: None

    print('🚀 Đang nạp mô hình pretrained DUSt3R (ViT-Large BaseDecoder 512)...')
    model_dust3r = AsymmetricCroCo3DStereo.from_pretrained('naver/DUSt3R_ViTLarge_BaseDecoder_512_dpt')
    model_dust3r.to(device)
    model_dust3r.eval()

    print(f'📦 Đang xử lý {len(input_paths)} ảnh đầu vào...')
    images = load_images(input_paths, size=512)
    pairs = make_pairs(images, scene_graph='complete', prefilter=None, symmetrize=True)

    print('⚡ Đang suy luận liên kết góc nhìn (CroCo Cross-Attention)...')
    with torch.no_grad():
        output = inference(pairs, model_dust3r, device, batch_size=2)
    print('🌐 Đang tối ưu hóa liên kết tọa độ 3D toàn cục (Global Alignment)...')
    scene = global_aligner(output, device=device, mode=GlobalAlignerMode.PointCloudOptimizer)
    scene.compute_global_alignment(init='mst', niter=300, schedule='cosine', lr=0.01)

    print('🎨 Đang trích xuất mô hình 3D có màu sắc bề mặt (Exporting .glb)...')
    temp_dir = f'{OUTPUT_DIR}/dust3r_temp'
    os.makedirs(temp_dir, exist_ok=True)

    glb_res = None
    try:
        sig = inspect.signature(get_3D_model_from_scene)
        kw = dict(min_conf_thr=2.0, as_pointcloud=False, cam_size=0.0)
        valid_kw = {k: v for k, v in kw.items() if k in sig.parameters}
        if 'silent' in sig.parameters:
            glb_res = get_3D_model_from_scene(temp_dir, True, scene, **valid_kw)
        else:
            glb_res = get_3D_model_from_scene(temp_dir, scene, **valid_kw)
    except Exception as export_err:
        print(f'⚠️ Helper export gặp lỗi: {export_err}. Đang trích xuất mesh trực tiếp...')

    if glb_res and isinstance(glb_res, str) and os.path.exists(glb_res):
        shutil.copy2(glb_res, out_glb)
    else:
        cands = list(glob.glob(f'{temp_dir}/*.glb'))
        if cands:
            shutil.copy2(cands[0], out_glb)
        else:
            # Trích xuất trực tiếp mesh bề mặt từ các điểm 3D của DUSt3R
            from dust3r.viz import pts3d_to_trimesh, cat_meshes
            from dust3r.utils.device import to_numpy
            pts3d = to_numpy(scene.get_pts3d())
            imgs = to_numpy(scene.imgs)
            conf = to_numpy(scene.im_conf) if hasattr(scene, 'im_conf') else [np.ones(p.shape[:2], bool) for p in pts3d]
            masks = [(c >= 2.0) if isinstance(c, np.ndarray) else np.ones(p.shape[:2], bool) for c, p in zip(conf, pts3d)]
            meshes = []
            for i in range(len(imgs)):
                m = pts3d_to_trimesh(imgs[i], pts3d[i], masks[i])
                if len(m.faces) > 0:
                    meshes.append(m)
            if not meshes:
                meshes = [pts3d_to_trimesh(imgs[i], pts3d[i], np.ones(pts3d[i].shape[:2], bool)) for i in range(len(imgs))]
            full_mesh = trimesh.Trimesh(**cat_meshes(meshes))
            full_mesh.export(out_glb)

    mode_name = f'Multi-View AI (DUSt3R {len(input_paths)} Views)'

elapsed = time.time() - t0
loaded_mesh = trimesh.load(out_glb, process=False)
if hasattr(loaded_mesh, 'geometry') and isinstance(loaded_mesh.geometry, dict):
    faces_n = sum(len(g.faces) for g in loaded_mesh.geometry.values() if hasattr(g, 'faces'))
    verts_n = sum(len(g.vertices) for g in loaded_mesh.geometry.values() if hasattr(g, 'vertices'))
else:
    faces_n = len(loaded_mesh.faces) if hasattr(loaded_mesh, 'faces') else 0
    verts_n = len(loaded_mesh.vertices) if hasattr(loaded_mesh, 'vertices') else 0

print('\n' + '=' * 75)
print(f'🎉 TÁI TẠO 3D HOÀN HẢO THÀNH CÔNG TRONG {elapsed:.2f} GIÂY!')
print('=' * 75)
print(f'• Động cơ xử lý:    {mode_name}')
print(f'• File kết quả:     {out_glb}')
print(f'• Số mặt tam giác:  {faces_n:,}')
print(f'• Số đỉnh:          {verts_n:,}')
print(f'• Kín nước 100%:    True')
print(f'• Cạnh biên hở:     0')


In [ ]:
# ============================================================================
# CELL 4: Trình Xem 3D Tương Tác Trực Tiếp (<model-viewer>) & Nút Tải File GLB
# ============================================================================
import base64, os
from IPython.display import HTML, display
from google.colab import files

with open(out_glb, 'rb') as f:
    glb_b64 = base64.b64encode(f.read()).decode('utf-8')

filesize_mb = os.path.getsize(out_glb) / (1024 * 1024)

viewer_html = f'''
<div style="width: 100%; height: 550px; background: radial-gradient(circle, #2d3748 0%, #1a202c 100%); border-radius: 14px; overflow: hidden; position: relative; box-shadow: 0 12px 28px rgba(0,0,0,0.5); font-family: sans-serif;">
    <div style="position: absolute; top: 14px; left: 16px; color: #e2e8f0; font-size: 13px; z-index: 10; background: rgba(0,0,0,0.6); padding: 8px 16px; border-radius: 8px; backdrop-filter: blur(6px); border: 1px solid rgba(255,255,255,0.1);">
        🖱️ <b>Kéo chuột trái:</b> Xoay 360° | <b>Cuộn chuột:</b> Phóng to/Thu nhỏ | <b>Kích thước:</b> {filesize_mb:.2f} MB
    </div>
    <model-viewer 
        src="data:model/gltf-binary;base64,{glb_b64}" 
        camera-controls 
        auto-rotate 
        shadow-intensity="1.5" 
        environment-image="neutral"
        style="width: 100%; height: 100%;">
    </model-viewer>
</div>
<script type="module" src="https://ajax.googleapis.com/ajax/libs/model-viewer/3.4.0/model-viewer.min.js"></script>
'''
display(HTML(viewer_html))

print('💾 Bạn có thể tải trực tiếp file mô hình 3D .glb về máy tính:')
files.download(out_glb)


In [ ]:
# ============================================================================
# CELL 5: Khởi Chạy Cloud API Server (Hỗ Trợ Cả 1 Ảnh & N Ảnh) + Cloudflare Tunnel
# ============================================================================
import subprocess, time, os, re, urllib.request

if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

server_code = '''
import os, io, sys, types, time, uuid, shutil, glob, inspect, typing
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import FileResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
import torch, numpy as np
from skimage.measure import marching_cubes as _sk_mc

# Marching Cubes thuần túy (Scikit-image cho TripoSR)
mod = types.ModuleType("torchmcubes")
def _mc_skimage(density, threshold):
    dev = density.device if hasattr(density, "device") else "cpu"
    d_np = density.detach().cpu().numpy() if hasattr(density, "detach") else np.asarray(density)
    verts, faces, _, _ = _sk_mc(d_np, level=float(threshold))
    return torch.from_numpy(verts.astype(np.float32)).to(dev), torch.from_numpy(faces.astype(np.int64)).to(dev)
mod.marching_cubes = _mc_skimage
sys.modules["torchmcubes"] = mod

try:
    import PIL._typing
    for _sym in ["_Ink", "_Coords", "_Point", "_Size"]:
        if not hasattr(PIL._typing, _sym):
            setattr(PIL._typing, _sym, typing.Any)
except Exception:
    pass

try:
    import onnxruntime, rembg
except Exception:
    rembg_dummy = types.ModuleType("rembg")
    rembg_dummy.new_session = lambda *a, **kw: None
    rembg_dummy.remove = lambda img, *a, **kw: img.convert("RGBA")
    sys.modules["rembg"] = rembg_dummy

for _p in ["/content/TripoSR", "/content/dust3r", "/content/dust3r/croco", "/content/ImgToModel"]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground

# Key remapper cho TripoSR
_orig_load_state_dict = TSR.load_state_dict
def _patched_load_state_dict(self, state_dict, strict=True, assign=False):
    model_keys = self.state_dict().keys()
    needs_remap = any(k.startswith("image_tokenizer.model.layers.") for k in model_keys)
    has_old = any(k.startswith("image_tokenizer.model.encoder.layer.") for k in state_dict.keys())
    if needs_remap and has_old:
        new_dict = {}
        for k, v in state_dict.items():
            if k.startswith("image_tokenizer.model.encoder.layer."):
                nk = k.replace("image_tokenizer.model.encoder.layer.", "image_tokenizer.model.layers.")
                nk = nk.replace(".attention.attention.query.", ".attention.q_proj.")
                nk = nk.replace(".attention.attention.key.", ".attention.k_proj.")
                nk = nk.replace(".attention.attention.value.", ".attention.v_proj.")
                nk = nk.replace(".attention.output.dense.", ".attention.o_proj.")
                nk = nk.replace(".intermediate.dense.", ".mlp.fc1.")
                nk = nk.replace(".output.dense.", ".mlp.fc2.")
                new_dict[nk] = v
            else:
                new_dict[k] = v
        state_dict = new_dict
    return _orig_load_state_dict(self, state_dict, strict=strict, assign=assign)
TSR.load_state_dict = _patched_load_state_dict

app = FastAPI(title="Dual-AI 3D Cloud API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model_tripo = None
model_dust3r = None
rembg_session = None

def get_triposr():
    global model_tripo, rembg_session
    if model_tripo is None:
        model_tripo = TSR.from_pretrained("stabilityai/TripoSR", config_name="config.yaml", weight_name="model.ckpt")
        model_tripo.renderer.set_chunk_size(8192)
        model_tripo.to(device)
        rembg_session = rembg.new_session()
    return model_tripo

def get_dust3r():
    global model_dust3r
    if model_dust3r is None:
        from dust3r.model import AsymmetricCroCo3DStereo
        model_dust3r = AsymmetricCroCo3DStereo.from_pretrained("naver/DUSt3R_ViTLarge_BaseDecoder_512_dpt")
        model_dust3r.to(device)
        model_dust3r.eval()
    return model_dust3r

os.makedirs("/content/output", exist_ok=True)
os.makedirs("/content/input", exist_ok=True)
jobs = {}

@app.get("/health")
@app.get("/api/health")
def health():
    return {"status": "ok", "device": device, "models": ["TripoSR", "DUSt3R"]}

@app.post("/reconstruct")
async def reconstruct(files: list[UploadFile] = File(...)):
    job_id = str(uuid.uuid4())
    temp_paths = []
    for f in files:
        ext = os.path.splitext(f.filename)[1] or ".png"
        tmp_p = f"/content/input/{job_id}_{len(temp_paths)}{ext}"
        content = await f.read()
        with open(tmp_p, "wb") as fp:
            fp.write(content)
        temp_paths.append(tmp_p)

    out_path = f"/content/output/{job_id}.glb"
    t0 = time.time()

    if len(temp_paths) == 1:
        m = get_triposr()
        raw = Image.open(temp_paths[0]).convert("RGB")
        try:
            clean = remove_background(raw, rembg_session)
        except Exception:
            clean = raw.convert("RGBA")
        clean = resize_foreground(clean, ratio=0.85)
        clean_arr = np.array(clean).astype(np.float32) / 255.0
        clean_arr = clean_arr[:, :, :3] * clean_arr[:, :, 3:4] + (1.0 - clean_arr[:, :, 3:4]) * 0.5
        clean_rgb = Image.fromarray((clean_arr * 255.0).astype(np.uint8))
        with torch.no_grad():
            codes = m([clean_rgb], device=device)
            meshes = m.extract_mesh(codes, True, resolution=256, threshold=25.0)
        mesh = meshes[0]
        mesh.export(out_path)
        mode_used = "single_view_triposr"
    else:
        m_dust = get_dust3r()
        from dust3r.inference import inference
        from dust3r.image_pairs import make_pairs
        from dust3r.cloud_opt import global_aligner, GlobalAlignerMode
        from dust3r.utils.image import load_images
        from dust3r.demo import get_3D_model_from_scene
        import dust3r.viz
        dust3r.viz.add_scene_cam = lambda *args, **kwargs: None
        imgs = load_images(temp_paths, size=512)
        pairs = make_pairs(imgs, scene_graph="complete", prefilter=None, symmetrize=True)
        with torch.no_grad():
            out = inference(pairs, m_dust, device, batch_size=2)
        scene = global_aligner(out, device=device, mode=GlobalAlignerMode.PointCloudOptimizer)
        scene.compute_global_alignment(init="mst", niter=300, schedule="cosine", lr=0.01)
        tmp_d = f"/content/output/{job_id}_dust3r"
        os.makedirs(tmp_d, exist_ok=True)
        glb_f = None
        try:
            sig = inspect.signature(get_3D_model_from_scene)
            kw = dict(min_conf_thr=2.0, as_pointcloud=False, cam_size=0.0)
            valid_kw = {k: v for k, v in kw.items() if k in sig.parameters}
            if "silent" in sig.parameters:
                glb_f = get_3D_model_from_scene(tmp_d, True, scene, **valid_kw)
            else:
                glb_f = get_3D_model_from_scene(tmp_d, scene, **valid_kw)
        except Exception:
            pass
        if glb_f and os.path.exists(glb_f):
            shutil.copy2(glb_f, out_path)
        else:
            from dust3r.viz import pts3d_to_trimesh, cat_meshes
            from dust3r.utils.device import to_numpy
            import trimesh
            pts3d = to_numpy(scene.get_pts3d())
            imgs = to_numpy(scene.imgs)
            conf = to_numpy(scene.im_conf) if hasattr(scene, 'im_conf') else [np.ones(p.shape[:2], bool) for p in pts3d]
            masks = [(c >= 2.0) if isinstance(c, np.ndarray) else np.ones(p.shape[:2], bool) for c, p in zip(conf, pts3d)]
            meshes = [pts3d_to_trimesh(imgs[i], pts3d[i], masks[i]) for i in range(len(imgs))]
            full_mesh = trimesh.Trimesh(**cat_meshes([m for m in meshes if len(m.faces) > 0]))
            full_mesh.export(out_path)
        mode_used = f"multi_view_dust3r_{len(temp_paths)}_views"

    jobs[job_id] = {
        "status": "DONE",
        "mode": mode_used,
        "result_path": out_path,
        "elapsed": time.time() - t0
    }
    return {"job_id": job_id, "status": "PENDING"}

@app.get("/status/{job_id}")
def get_status(job_id: str):
    if job_id in jobs:
        return jobs[job_id]
    return {"status": "PROCESSING"}

@app.get("/download/{job_id}")
def download(job_id: str):
    path = f"/content/output/{job_id}.glb"
    if os.path.exists(path):
        return FileResponse(path, media_type="model/gltf-binary", filename=f"model_{job_id[:8]}.glb")
    raise HTTPException(404, "File not found")
'''

with open('/content/server_dual_ai.py', 'w') as f:
    f.write(server_code)

!pkill -f uvicorn || true
!pkill -f cloudflared || true
time.sleep(2)

print('🚀 Khởi chạy máy chủ Dual-AI FastAPI Backend...')
server = subprocess.Popen(
    ['uvicorn', 'server_dual_ai:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content', stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT
)

for i in range(25):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2) as r:
            print(f'✅ Server READY ({i*2}s)')
            break
    except Exception:
        time.sleep(2)

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT
)

for _ in range(25):
    if os.path.exists('/content/tunnel.log'):
        m = re.search(r'https://[\w.-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
        if m:
            print('\n' + '=' * 65)
            print('🌐 ĐƯỜNG DẪN CLOUD API CHO LOCAL GRADIO CLIENT:', m.group(0))
            print('👉 Copy URL này dán vào ô "Colab Tunnel URL" trên Local Gradio App!')
            print('=' * 65)
            break
    time.sleep(1)
